In [1]:
import pandas as pd
import numpy as np


In [3]:
from pathlib import Path

In [4]:
dataset = Path('/kaggle/input/datasets/himanshuydv11/facial-emotion-dataset/facial_emotion_dataset/dataset')

emotion_df = dataset

In [5]:
emotion_df

PosixPath('/kaggle/input/datasets/himanshuydv11/facial-emotion-dataset/facial_emotion_dataset/dataset')

In [8]:
import os 


In [10]:
emotion_recognition = []




def convert_df(folder_Path):
    for folder_name in os.listdir(folder_Path):
        folder_path = os.path.join(folder_Path, folder_name)
    
        if os.path.isdir(folder_path):  # Check if it is a directory
            for image_name in os.listdir(folder_path):
                if image_name.endswith(('.png', '.jpg', '.jpeg')):  # Filter image types
                    image_path = os.path.join(folder_path, image_name)
                    if folder_Path == emotion_df:
                        emotion_recognition.append({'image_path': image_path, 'label': folder_name})





emotion_df = convert_df(emotion_df)
emotion_df = pd.DataFrame(emotion_recognition)

In [11]:
emotion_df

,image_path,label
0,/kaggle/input/datasets/himanshuydv11/facial-em...,Surprise
1,/kaggle/input/datasets/himanshuydv11/facial-em...,Surprise
2,/kaggle/input/datasets/himanshuydv11/facial-em...,Surprise
3,/kaggle/input/datasets/himanshuydv11/facial-em...,Surprise
4,/kaggle/input/datasets/himanshuydv11/facial-em...,Surprise
...,...,...
15147,/kaggle/input/datasets/himanshuydv11/facial-em...,Ahegao
15148,/kaggle/input/datasets/himanshuydv11/facial-em...,Ahegao
15149,/kaggle/input/datasets/himanshuydv11/facial-em...,Ahegao
15150,/kaggle/input/datasets/himanshuydv11/facial-em...,Ahegao


In [12]:
import pandas as pd
import numpy as np
import cv2
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical


2026-02-17 12:22:41.661255: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771330961.913625      56 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771330961.989388      56 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771330962.585809      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771330962.585862      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771330962.585866      56 computation_placer.cc:177] computation placer alr

In [13]:
def load_and_preprocess_data(emotion_df, input_width, input_height):
    images = []
    labels = []
    
    for index, row in emotion_df.iterrows():
        image_path = row['image_path']
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
        image = cv2.resize(image, (input_width, input_height))  # Resize
        
        images.append(image)
        labels.append(row['label'])
    
    images = np.array(images) / 255.0  # Normalize images
    images = np.expand_dims(images, axis=-1)  # Add channels dimension
    
    return images, np.array(labels)

# Example DataFrame
# emotion_df = pd.read_csv('path_to_your_data.csv')  # Uncomment and set the correct path
#emotion_df = pd.DataFrame({'image_path': ['image1.jpg', 'image2.jpg'], 
                        #   'label': ['surprised', 'neutral']})  # Replace with your actual data


In [14]:
def create_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # Adjust for classification
    return model


In [15]:
# Preparing the dataset
input_width, input_height = 200, 200  # Modify as needed
images, labels = load_and_preprocess_data(emotion_df, input_width, input_height)

# Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)

# Split the data
X_train, X_val, y_train, y_val = train_test_split(images, labels_categorical, test_size=0.2, random_state=42)

# Build the model
model = create_model((input_width, input_height, 1), len(le.classes_))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-17 12:26:54.624867: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 464s 1s/step - accuracy: 0.3350 - loss: 1.9526 - val_accuracy: 0.4731 - val_loss: 1.2848
Epoch 2/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 490s 1s/step - accuracy: 0.6509 - loss: 0.8874 - val_accuracy: 0.5582 - val_loss: 1.1006
Epoch 4/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 464s 1s/step - accuracy: 0.7607 - loss: 0.6280 - val_accuracy: 0.5612 - val_loss: 1.1796
Epoch 5/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 466s 1s/step - accuracy: 0.8685 - loss: 0.3771 - val_accuracy: 0.5589 - val_loss: 1.3659
Epoch 6/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 476s 1s/step - accuracy: 0.9402 - loss: 0.1901 - val_accuracy: 0.5318 - val_loss: 1.8574
Epoch 7/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 480s 1s/step - accuracy: 0.9744 - loss: 0.0937 - val_accuracy: 0.5460 - val_loss: 2.4468
Epoch 8/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 465s 1s/step - accuracy: 0.9878 - loss: 0.0493 - val_accuracy: 0.5457 - val_loss: 2.6700
Epoch 9/10
379/379 ━━━━━━━━━━━━━━━━━━━━ 494s 1s/step - accuracy: 0.9909 - loss: 0.0360 - val_accu

In [16]:
loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation Accuracy: {accuracy:.2f}')


95/95 ━━━━━━━━━━━━━━━━━━━━ 27s 287ms/step - accuracy: 0.5251 - loss: 2.9275
Validation Accuracy: 0.53


In [17]:
model.save('emotion_model.h5')


In [1]:
import numpy as np
import cv2
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt


2026-02-17 14:40:12.553157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771339212.808584      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771339212.883377      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771339213.486426      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771339213.486482      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771339213.486486      55 computation_placer.cc:177] computation placer alr

In [2]:
# Load your trained model
model_path = '/kaggle/working/emotion_model.h5'  # Path to your .h5 model
model = load_model(model_path)

# Load the label encoder fitted during training
le = LabelEncoder()
# Fit the label encoder again if you have the unique labels
# For example: le.fit(['surprised', 'neutral', 'angry', 'sad'])
# Alternatively, you could store the classes during training and load them here


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/kaggle/working/emotion_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
def preprocess_image(image_path, input_width, input_height):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    image = cv2.resize(image, (input_width, input_height))  # Resize

    # Normalize the image
    image = image / 255.0  # Scale pixel values to [0, 1]
    
    # Add channel dimension and batch dimension
    image = np.expand_dims(image, axis=-1)  # Shape: (height, width, 1)
    image = np.expand_dims(image, axis=0)   # Shape: (1, height, width, 1)
    
    return image


In [ ]:
def predict_emotion_on_image(model, image_path, input_width, input_height):
    processed_image = preprocess_image(image_path, input_width, input_height)
    
    # Make prediction
    predictions = model.predict(processed_image)
    # Get the predicted class index
    predicted_class_index = np.argmax(predictions[0])
    
    return le.inverse_transform([predicted_class_index])[0]  # Convert back to label

# Example usage with a list of test images
test_images = ['/kaggle/input/models/adityamodi20/face-images/tflite/default/1/pexels-adil-asainov-912950932-27967698.jpg', '/kaggle/input/models/adityamodi20/face-images/tflite/default/1/pexels-heitorverdifotos-2169434.jpg']  # Update with actual paths
input_width, input_height = 200, 200  # Ensure these match your model input dimensions

predictions = {}
for img in test_images:
    predicted_emotion = predict_emotion_on_image(model, img, input_width, input_height)
    predictions[img] = predicted_emotion

# Display results
for img_path, emotion in predictions.items():
    print(f'Image: {img_path} - Predicted Emotion: {emotion}')


In [ ]:
for img_path, emotion in predictions.items():
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB for display
    plt.imshow(img)
    plt.title(f'Predicted Emotion: {emotion}')
    plt.axis('off')
    plt.show()
